# Scenario Playground raw-data download

This notebook is the reproducible, instructional entry point for Step 3. It runs the same source-specific cache layer as the command-line runner. It preserves raw source files and metadata only; it does not merge sources, build features or targets, or train a model.

Before running the retrieval cell, confirm that the machine has the required packages and that any existing project credentials are configured. Credential values are never printed or copied into metadata.


## Workflow

1. Inspect the local storage root and configured source candidates.
2. Run selected source downloaders with independent files and sidecar metadata.
3. Reuse valid caches unless refresh is explicitly enabled.
4. Inspect the generated coverage report and failures.

The canonical non-notebook command is python -m scenario_modeling.run_downloads. Use refresh=True only when an explicit source refresh is intended.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scenario_modeling").exists():
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.config import DATA_ROOT, RAW_SOURCE_DIRS, ensure_storage

ensure_storage()
print(f"data_root={DATA_ROOT}")
print("raw_sources=")
for name, path in RAW_SOURCE_DIRS.items():
    print(f"  {name}: {path}")


## Select and run the raw-data retrievals

Each source downloader is independent. A failed source writes a separate failure record and leaves an existing valid cache untouched. The default below is intentionally disabled so that opening the notebook does not unexpectedly make network requests. Set RUN_DOWNLOADS to True for an explicit retrieval run.


In [ ]:
from scenario_modeling.run_downloads import run_downloads

SOURCES = [
    "yahoo",
    "fred",
    "alfred",
    "federal_reserve",
    "bls",
    "bea",
    "cleveland_fed",
    "atlanta_fed",
    "ism",
    "kansas_city_fed",
    "philadelphia_fed",
]
RUN_DOWNLOADS = False
REFRESH = False

if RUN_DOWNLOADS:
    results = run_downloads(refresh=REFRESH, only=SOURCES)
    print(f"completed_attempts={len(results)}")
else:
    print("Set RUN_DOWNLOADS=True to execute controlled source retrievals.")


## Inspect metadata and coverage

The generated report is the Step 3 handoff. It distinguishes successful downloads, downloads with source gaps, reused caches, and source failures. The fields are descriptive only: no missing-value treatment or time alignment is performed here.


In [ ]:
from scenario_modeling.src.coverage import load_failures, load_metadata, write_report

metadata_records = load_metadata()
failure_records = load_failures()
report_path = write_report()
print(f"metadata_records={len(metadata_records)}")
print(f"failure_records={len(failure_records)}")
print(f"report_path={report_path}")

CORRECTED_DATASETS = {
    "kansas_city_fed_manufacturing_pmi_proxy_events",
    "fomc_policy_decisions",
    "ism_manufacturing_pmi_dbnomics",
}
print("corrected_step3_status=")
for record in metadata_records:
    if record.get("dataset_name") in CORRECTED_DATASETS:
        print(
            record.get("dataset_name"),
            record.get("retrieval_status"),
            record.get("selection_status", "selected-or-structured"),
            record.get("row_count"),
            record.get("minimum_observation_date"),
            record.get("maximum_observation_date"),
        )


## Checks before Step 4

- Confirm the Kansas City Fed manufacturing proxy is `PRIMARY_MVP_PROXY_READY_WITH_LIMITATIONS` and the DBnomics/ISM file is `REJECTED_PRIMARY_CANDIDATE`.
- Confirm `fomc_policy_decisions` contains scheduled holds and applicable unscheduled decisions, not only rate changes.
- Review the generated report's cached coverage, shared-period estimates, release-timing rules, and licensing limitations.
- Treat Step 3 as ready only when both corrected datasets have successful metadata sidecars; append-only failure records can describe earlier attempts.

Do not use this notebook to create shocks, forward-fill releases, construct the training matrix, or fit models; those decisions belong to later steps.
